# Building a Toy Cortex: EEG-like Oscillations from Spiking Neurons

**An undergraduate neuroscience practical**

Scalp EEG does not record single neurons — it records the summed electrical footprint of **thousands to millions of neurons** activating together, especially the synchronised synaptic currents of cortical pyramidal cells. The oscillations you see in an EEG trace (delta, theta, alpha, beta, gamma) emerge from the *interaction* of excitatory and inhibitory neurons in local circuits, not from any single cell "oscillating" on its own.

In this practical you will build that idea from the ground up:

1. Simulate a **single spiking neuron** using the Izhikevich model and see how four parameters reproduce very different real cortical firing patterns.
2. Wire **hundreds of excitatory and inhibitory neurons** together into a small recurrent network — a "toy cortex".
3. Sum the network's activity into a continuous, EEG-like signal.
4. Use spectral analysis to show that this signal contains genuine **oscillations**, and relate their frequency to the standard EEG bands.
5. Manipulate the excitation/inhibition balance and synaptic timing to shift the network between oscillation bands, and connect this to real mechanisms (e.g. PING gamma rhythms).

### How this practical works

- This notebook runs entirely in your browser (JupyterLite) — nothing is installed and nothing leaves your machine.
- Run cells from top to bottom with **Shift+Enter**. Later cells depend on earlier ones.
- Boxes marked **Your turn** ask you to change something and observe the result, or answer a short question. There is no single "correct" plot for these — we are looking for correct *reasoning* about what changed and why.
- The network simulations are small (hundreds of neurons, ~1 second of simulated time) so that they run in a few seconds in-browser. Real cortex has ~10<sup>10</sup> neurons — keep in mind everything here is a deliberately simplified teaching model, not a biophysically exact EEG simulator.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

%matplotlib inline

# A fixed random seed makes the network reproducible. Change it later if you
# want to see a different random network / trial.
RNG_SEED = 2026
np.random.seed(RNG_SEED)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Standard EEG frequency bands (Hz), used later to label the power spectrum
EEG_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta": (13, 30),
    "gamma": (30, 80),
}

## Part 1 — A single spiking neuron

The **Izhikevich model** (Izhikevich, 2003, *IEEE Trans. Neural Networks*) is a two-variable model that is cheap enough to simulate thousands of neurons, yet rich enough to reproduce the major firing patterns seen in real cortical recordings. It tracks membrane potential $v$ (mV) and a recovery variable $u$ that represents slow ionic currents (roughly, K$^+$ activation and Na$^+$ inactivation):

$$\frac{dv}{dt} = 0.04v^2 + 5v + 140 - u + I$$
$$\frac{du}{dt} = a(bv - u)$$

with the reset rule: whenever $v \geq 30$ mV, the neuron has fired a spike, and instantaneously

$$v \leftarrow c, \qquad u \leftarrow u + d$$

Four parameters control everything:

| Parameter | Biological meaning |
|---|---|
| $a$ | time scale of the recovery variable $u$ (smaller = slower recovery) |
| $b$ | sensitivity of $u$ to sub-threshold fluctuations of $v$ |
| $c$ | membrane potential immediately after a spike (reset value) |
| $d$ | how much the spike increments $u$ (spike-triggered adaptation) |

$I$ is the input current (synaptic drive + any injected current).

By choosing $a,b,c,d$, the *same two equations* reproduce very different cell types:

| Cell type | $a$ | $b$ | $c$ | $d$ | Notes |
|---|---|---|---|---|---|
| Regular spiking (RS) — typical excitatory cortical pyramidal cell | 0.02 | 0.2 | -65 | 8 | fires, then adapts (slows down) |
| Intrinsically bursting (IB) | 0.02 | 0.2 | -55 | 4 | fires a burst, then regular spikes |
| Chattering (CH) | 0.02 | 0.2 | -50 | 2 | repetitive high-frequency bursts |
| Fast spiking (FS) — typical cortical inhibitory interneuron | 0.1 | 0.2 | -65 | 2 | little/no adaptation, can fire very fast |
| Low-threshold spiking (LTS) | 0.02 | 0.25 | -65 | 2 | inhibitory, rebound bursts |

RS and FS are the two cell types we will use to build the network in Part 2: RS neurons as our excitatory ("pyramidal-like") population, FS neurons as our inhibitory population — this excitatory/inhibitory (E/I) pairing is what generates network oscillations.

In [ ]:
def simulate_izhikevich_neuron(a, b, c, d, I, T=300, dt=0.5, v0=-65):
    """Simulate a single Izhikevich neuron driven by constant current I.

    Parameters
    ----------
    a, b, c, d : float
        Izhikevich model parameters (see table above).
    I : float
        Constant input current.
    T : float
        Total simulated duration in ms.
    dt : float
        Integration time step in ms (0.5 ms is a good default for this model).
    v0 : float
        Initial membrane potential in mV.

    Returns
    -------
    t : (n,) array of time points in ms
    v : (n,) array of membrane potential in mV (spikes clipped to 30 mV for plotting)
    spike_times : array of times (ms) at which the neuron fired
    """
    n_steps = int(T / dt)
    t = np.arange(n_steps) * dt
    v = np.zeros(n_steps)
    u = np.zeros(n_steps)
    v[0] = v0
    u[0] = b * v0
    spike_times = []

    for i in range(1, n_steps):
        v_prev, u_prev = v[i - 1], u[i - 1]
        dv = (0.04 * v_prev**2 + 5 * v_prev + 140 - u_prev + I) * dt
        du = a * (b * v_prev - u_prev) * dt
        v_new = v_prev + dv
        u_new = u_prev + du

        if v_new >= 30:
            spike_times.append(t[i])
            v_new = c
            u_new = u_new + d

        v[i] = v_new
        u[i] = u_new

    return t, v, np.array(spike_times)

In [ ]:
# Classic Izhikevich cell types (a, b, c, d, I)
cell_types = {
    "Regular spiking (RS)":       dict(a=0.02, b=0.2, c=-65, d=8, I=10),
    "Intrinsically bursting (IB)": dict(a=0.02, b=0.2, c=-55, d=4, I=10),
    "Chattering (CH)":            dict(a=0.02, b=0.2, c=-50, d=2, I=10),
    "Fast spiking (FS)":          dict(a=0.1,  b=0.2, c=-65, d=2, I=10),
    "Low-threshold spiking (LTS)": dict(a=0.02, b=0.25, c=-65, d=2, I=10),
}

fig, axes = plt.subplots(len(cell_types), 1, figsize=(9, 10), sharex=True)

for ax, (name, params) in zip(axes, cell_types.items()):
    t, v, spikes = simulate_izhikevich_neuron(**params, T=300)
    ax.plot(t, v, lw=1)
    ax.set_ylabel("v (mV)")
    ax.set_title(f"{name}  (a={params['a']}, b={params['b']}, c={params['c']}, d={params['d']}, I={params['I']})",
                 fontsize=9, loc="left")
    ax.set_ylim(-90, 40)

axes[-1].set_xlabel("time (ms)")
fig.tight_layout()
plt.show()

> ### 🧪 Your turn — Exercise 1
> Using the code cell below:
> 1. Take the **RS** parameters and try `I = 0, 2, 5, 20`. What happens to the firing rate? Is the relationship linear?
> 2. Take the **FS** parameters and compare its firing rate to RS at the *same* `I`. Which fires faster, and does it show adaptation (slowing down over time)? Relate this to the biological meaning of parameter `a`.
> 3. Set `I` to a small negative value (e.g. `-5`) for RS. What happens, and why does this make biological sense for an input *current*?
>
> Answer in 2–3 sentences per question in the markdown cell provided (double-click "*Your answer here*" to edit it).

In [ ]:
# Scaffold for Exercise 1 — change the parameters and re-run
a, b, c, d, I = 0.02, 0.2, -65, 8, 10  # RS parameters to start

t, v, spikes = simulate_izhikevich_neuron(a, b, c, d, I, T=300)

fig, ax = plt.subplots()
ax.plot(t, v, lw=1)
ax.set_xlabel("time (ms)")
ax.set_ylabel("v (mV)")
ax.set_title(f"a={a}, b={b}, c={c}, d={d}, I={I}  →  {len(spikes)} spikes in {t[-1]:.0f} ms "
             f"({1000 * len(spikes) / t[-1]:.1f} Hz)")
plt.show()

*Your answer here:*

1.
2.
3.

## Part 2 — From single neurons to a network "EEG"

A single neuron's membrane potential is *not* an EEG signal. EEG electrodes on the scalp pick up the summed extracellular currents from huge, synchronised populations of cortical pyramidal neurons — mostly the synaptic currents flowing into their dendrites. A population only produces a visible rhythm at the scalp if its neurons fire in a coordinated, oscillatory way, rather than independently at random times.

We will now build a small recurrent network — a "toy cortex" — following the classic design of Izhikevich (2003):

- **800 excitatory neurons** (RS parameters) — analogous to cortical pyramidal cells.
- **200 inhibitory neurons** (FS parameters) — analogous to fast-spiking interneurons.
- Neurons are connected **randomly and sparsely**: every neuron receives input from a random subset of the whole population. Excitatory synapses are positive (depolarising); inhibitory synapses are negative (hyperpolarising).
- Each neuron also receives noisy background input, representing all the other brain activity we are not explicitly modelling.
- Synaptic input travels with a small **transmission delay** (separately tunable for excitatory and inhibitory connections) — this delay turns out to be critical for setting the oscillation frequency (Part 3).

**Our EEG-like signal (the "proxy")** is the *sum, across all excitatory neurons, of the synaptic current they receive at each time step*. This is a simplified stand-in for the local field potential: real LFP/EEG is dominated by summed synaptic currents in pyramidal cells, so summing our model's excitatory synaptic drive is a reasonable (if crude) analogue. We will also look at the **population firing rate** (a histogram of all spikes across the network) as a second, complementary view of the same underlying rhythm.

In [ ]:
def run_network(Ne=800, Ni=200, T=1000, p_connect=1.0,
                 exc_strength=0.5, inh_strength=1.0,
                 drive_E=5.0, drive_I=2.0,
                 delay_E=1, delay_I=1, seed=None):
    """Simulate a recurrent network of excitatory (RS) and inhibitory (FS)
    Izhikevich neurons for T milliseconds (1 ms time bins, following
    Izhikevich 2003).

    Parameters
    ----------
    Ne, Ni : number of excitatory / inhibitory neurons.
    T : simulated duration in ms.
    p_connect : probability that any given neuron synapses onto any other
        (1.0 = fully connected, as in the classic Izhikevich 2003 demo).
    exc_strength, inh_strength : maximum synaptic weight for excitatory /
        inhibitory connections (inhibitory weights are negative).
    drive_E, drive_I : standard deviation (in current units) of independent
        background noise driving each population — this stands in for all
        the input the network receives from the rest of the brain.
    delay_E, delay_I : synaptic transmission delay (ms) for excitatory /
        inhibitory connections. This is the key knob explored in Part 3.
    seed : optional random seed for reproducibility.

    Returns a dict with:
      firings : (n_spikes, 2) array of [time_ms, neuron_id] (neuron_id < Ne
                is excitatory, >= Ne is inhibitory)
      proxy   : (T,) array, our EEG-like signal — total synaptic current
                delivered to the excitatory population at each ms
      rate    : (T,) array, total number of spikes fired (any neuron) at
                each ms — a simple population multi-unit-activity signal
      Ne, Ni, T : as given
    """
    if seed is not None:
        np.random.seed(seed)

    N = Ne + Ni

    # Heterogeneous RS / FS parameters, as in Izhikevich (2003)
    re, ri = np.random.rand(Ne), np.random.rand(Ni)
    a = np.concatenate([0.02 * np.ones(Ne), 0.02 + 0.08 * ri])
    b = np.concatenate([0.2 * np.ones(Ne), 0.25 - 0.05 * ri])
    c = np.concatenate([-65 + 15 * re**2, -65 * np.ones(Ni)])
    d = np.concatenate([8 - 6 * re**2, 2 * np.ones(Ni)])

    # Random sparse connectivity. Columns 0:Ne are excitatory synapses
    # (positive weight), columns Ne:N are inhibitory (negative weight).
    conn_mask = np.random.rand(N, N) < p_connect
    np.fill_diagonal(conn_mask, False)
    weights = np.zeros((N, N))
    weights[:, :Ne] = exc_strength * np.random.rand(N, Ne)
    weights[:, Ne:] = -inh_strength * np.random.rand(N, Ni)
    S = weights * conn_mask

    v = -65 * np.ones(N)
    u = b * v

    # Ring buffer holding synaptic input that is "in transit" and will
    # arrive at its target after the appropriate delay.
    buffer_len = max(delay_E, delay_I) + 1
    pending = np.zeros((buffer_len, N))

    spike_t, spike_id = [], []
    proxy_signal = np.zeros(T)
    rate_signal = np.zeros(T)

    for t in range(T):
        idx = t % buffer_len
        I_syn = pending[idx].copy()
        pending[idx] = 0.0

        I_noise = np.concatenate([drive_E * np.random.randn(Ne),
                                   drive_I * np.random.randn(Ni)])
        I = I_syn + I_noise

        fired = np.where(v >= 30)[0]
        if fired.size:
            spike_t.extend([t] * fired.size)
            spike_id.extend(fired.tolist())
            v[fired] = c[fired]
            u[fired] = u[fired] + d[fired]

            fired_E = fired[fired < Ne]
            fired_I = fired[fired >= Ne]
            if fired_E.size:
                pending[(t + delay_E) % buffer_len] += S[:, fired_E].sum(axis=1)
            if fired_I.size:
                pending[(t + delay_I) % buffer_len] += S[:, fired_I].sum(axis=1)

        rate_signal[t] = fired.size
        proxy_signal[t] = I_syn[:Ne].sum()  # synaptic drive onto excitatory (pyramidal-like) cells

        # Two 0.5 ms half-steps for numerical stability (standard for this model)
        v = v + 0.5 * (0.04 * v**2 + 5 * v + 140 - u + I)
        v = v + 0.5 * (0.04 * v**2 + 5 * v + 140 - u + I)
        u = u + a * (b * v - u)

    firings = np.column_stack([spike_t, spike_id]) if spike_t else np.empty((0, 2))
    return dict(firings=firings, proxy=proxy_signal, rate=rate_signal, Ne=Ne, Ni=Ni, T=T)

In [ ]:
import time

t0 = time.time()
net = run_network(seed=RNG_SEED)  # default "toy cortex", 1000 ms
print(f"Simulated {net['T']} ms of a {net['Ne'] + net['Ni']}-neuron network "
      f"in {time.time() - t0:.1f} s, {len(net['firings'])} spikes total.")

In [ ]:
from scipy.ndimage import gaussian_filter1d

def plot_raster_and_population(net, smooth_sigma_ms=3, title=None):
    """Two-panel figure: spike raster (top) and the network's summed/averaged
    population activity (bottom) — the emergent EEG-like oscillation.
    """
    firings, Ne, Ni, T = net["firings"], net["Ne"], net["Ni"], net["T"]

    fig, (ax_raster, ax_pop) = plt.subplots(
        2, 1, figsize=(10, 6), sharex=True,
        gridspec_kw={"height_ratios": [2, 1]},
    )

    # --- Top: raster plot, excitatory and inhibitory neurons colour-coded ---
    if firings.size:
        is_exc = firings[:, 1] < Ne
        ax_raster.scatter(firings[is_exc, 0], firings[is_exc, 1],
                           s=1, c="tab:blue", label=f"excitatory (n={Ne})")
        ax_raster.scatter(firings[~is_exc, 0], firings[~is_exc, 1],
                           s=1, c="tab:red", label=f"inhibitory (n={Ni})")
    ax_raster.set_ylabel("neuron #")
    ax_raster.set_title(title or "Network raster and population activity")
    ax_raster.legend(loc="upper right", markerscale=8, fontsize=8)

    # --- Bottom: population activity ---
    # Solid line: our EEG-like proxy (summed excitatory synaptic current).
    # Dashed line: population firing rate (spikes/ms, all neurons), rescaled
    # for display — a cross-check that both views show the same rhythm.
    t_axis = np.arange(T)
    proxy_smooth = gaussian_filter1d(net["proxy"], sigma=smooth_sigma_ms)
    rate_smooth = gaussian_filter1d(net["rate"], sigma=smooth_sigma_ms)

    ax_pop.plot(t_axis, proxy_smooth, color="k", lw=1.2, label="EEG-like proxy (Σ synaptic current)")
    ax_rate = ax_pop.twinx()
    ax_rate.plot(t_axis, rate_smooth, color="tab:green", lw=1, alpha=0.7, label="population firing rate")
    ax_rate.set_ylabel("spikes / ms", color="tab:green")
    ax_rate.tick_params(axis="y", labelcolor="tab:green")

    ax_pop.set_xlabel("time (ms)")
    ax_pop.set_ylabel("Σ synaptic current (a.u.)")
    lines1, labels1 = ax_pop.get_legend_handles_labels()
    lines2, labels2 = ax_rate.get_legend_handles_labels()
    ax_pop.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)

    fig.tight_layout()
    return fig


plot_raster_and_population(net, title="Default toy cortex (Ne=800, Ni=200)")
plt.show()

### Is that actually an oscillation?

The population trace above looks "wavy", but eyeballing a time series is not enough — real EEG analysis quantifies rhythms with the **power spectrum**, which shows how much of the signal's variance is explained by each frequency. A genuine oscillation shows up as a clear peak; noise without rhythmic structure gives a flat, featureless spectrum.

We use `scipy.signal.welch`, which splits the signal into overlapping segments, computes a periodogram for each, and averages them — a standard, robust way to estimate power spectral density (PSD) from a noisy, finite-length signal (exactly what EEG analysts do with real recordings).

In [ ]:
def plot_power_spectrum(net, fs=1000, fmax=100, ax=None, label=None):
    """Welch power spectral density of the network's EEG-like proxy signal,
    with standard EEG bands shaded for reference. fs = sampling rate in Hz
    (1000 Hz because our proxy signal has one sample per simulated ms).
    """
    freqs, psd = signal.welch(net["proxy"], fs=fs, nperseg=min(256, net["T"]))
    mask = freqs <= fmax

    if ax is None:
        fig, ax = plt.subplots()
        band_colors = plt.cm.Pastel1(np.linspace(0, 1, len(EEG_BANDS)))
        for (band, (lo, hi)), color in zip(EEG_BANDS.items(), band_colors):
            ax.axvspan(lo, hi, color=color, alpha=0.5, label=f"{band} ({lo}-{hi} Hz)")
        ax.legend(loc="upper right", fontsize=8, ncol=2)

    ax.plot(freqs[mask], psd[mask], color="k", lw=1.5, label=label)
    ax.set_xlabel("frequency (Hz)")
    ax.set_ylabel("power")
    ax.set_xlim(0, fmax)

    peak_freq = freqs[mask][np.argmax(psd[mask])]
    peak_band = next((band for band, (lo, hi) in EEG_BANDS.items() if lo <= peak_freq < hi), "outside defined bands")
    print(f"Peak oscillation frequency: {peak_freq:.1f} Hz  →  {peak_band} band")
    return freqs, psd, peak_freq


plot_power_spectrum(net)
plt.title("Power spectrum of the EEG-like proxy signal")
plt.show()

## Part 3 — Why does the network oscillate, and what sets the frequency?

The rhythm you just measured is not built into any single neuron — it emerges from a feedback loop:

1. Excitatory (RS) neurons receive enough noisy drive that some fire together.
2. Their spikes excite the inhibitory (FS) neurons, after a synaptic **delay**.
3. The inhibitory neurons fire back and suppress the excitatory population, again after a delay.
4. Inhibition decays away, excitation builds up again, and the cycle repeats.

This is the classic **PING mechanism** (Pyramidal-Interneuron Network Gamma; Tiesinga & Sejnowski, 2009) — one of the best-supported circuit explanations for cortical gamma oscillations. Its key prediction is that **the cycle length (and hence the oscillation frequency) is set largely by the excitation → inhibition → excitation loop delay**: a short loop delay produces fast oscillations (gamma, ~30-80 Hz); a longer delay produces slower ones (beta, ~13-30 Hz). Real gamma rhythms (~25-40 ms period) are consistent with fast GABA_A-mediated inhibition and short local axon conduction delays; slower rhythms in real cortex/thalamus (alpha, theta) typically also recruit additional slow currents and longer-range/thalamocortical loops that aren't in this simplified model — so don't expect our small local network to reach alpha or theta just by increasing delay. We can, however, clearly demonstrate the gamma → beta shift, which is the direct, well-established regime for this mechanism.

> ### 🧪 Your turn — Exercise 2
> The cell below runs the network at several different **inhibitory synaptic delays** (`delay_I`) and overlays their power spectra.
> 1. Run it as-is. Does increasing `delay_I` shift the peak frequency up or down? Does this match the PING prediction above?
> 2. Now edit `drive_E` (try doubling it, then halving it) with `delay_I` fixed. What happens to the peak frequency, and to the height/sharpness of the peak? What does a sharper, taller peak tell you about how synchronised the network is?
> 3. Try increasing `inh_strength` (e.g. to 2.0). Predict what will happen to the peak *before* you run it, then check.
> 4. In 2-3 sentences, relate what you found to what is known about real cortical E/I circuits and gamma-band oscillations.

In [ ]:
# Sweep inhibitory synaptic delay and compare power spectra.
# Tip: reduce T (e.g. to 500) while exploring if this runs too slowly for you.
delay_I_values = [1, 3, 6, 10]
drive_E = 5.0     # try changing this (Exercise 2, question 2)
inh_strength = 1.0  # try changing this (Exercise 2, question 3)

fig, ax = plt.subplots()
band_colors = plt.cm.Pastel1(np.linspace(0, 1, len(EEG_BANDS)))
for (band, (lo, hi)), color in zip(EEG_BANDS.items(), band_colors):
    ax.axvspan(lo, hi, color=color, alpha=0.4, label=f"{band} ({lo}-{hi} Hz)")

for delay_I in delay_I_values:
    net_sweep = run_network(T=1000, drive_E=drive_E, inh_strength=inh_strength,
                             delay_I=delay_I, seed=RNG_SEED)
    plot_power_spectrum(net_sweep, ax=ax, label=f"delay_I = {delay_I} ms")

ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.set_title("Effect of inhibitory synaptic delay on oscillation frequency")
plt.show()

*Your answer here:*

1.
2.
3.
4.

## Part 4 — How far can we trust this as a model of "EEG"?

This toy cortex reproduces one genuine and important phenomenon — that recurrent excitatory/inhibitory spiking networks generate emergent, tunable oscillations, and that synaptic timing and E/I balance are causal levers on frequency. That result generalises to real cortex. But treat the following as **real limitations**, not footnotes:

- **It is a proxy, not a volt on your scalp.** Real EEG also depends on the neurons' geometry (open- vs closed-field dipole arrangement), volume conduction through tissue/skull/scalp, and the orientation and alignment of thousands of cortical columns. Two networks with identical spiking statistics could look completely different at the scalp depending on this geometry — none of which is in our model.
- **~1,000 neurons vs ~10<sup>10</sup>.** We can show the *mechanism* at small scale, but real amplitude, and the precision of the frequency, depend on scale in ways a 1,000-neuron model cannot capture.
- **Single compartment, current-based synapses.** Real synapses have conductance dynamics, reversal potentials, receptor kinetics (fast GABA_A vs slow GABA_B, AMPA vs NMDA), and dendritic filtering — all of which shape real rhythms (e.g. GABA_B and NMDA are implicated in slower rhythms) and are entirely absent here.
- **No anatomy.** Real cortex has layered, distance-dependent, and long-range (including thalamocortical) connectivity; our network is randomly and uniformly connected.
- **One region, no closed loops.** Alpha rhythms in particular are strongly tied to thalamocortical loops, which we have not modelled at all — consistent with why our network could shift between gamma and beta but not cleanly reach alpha or theta.

> ### 💬 Discussion questions
> 1. If you wanted to make this model capable of generating a convincing alpha rhythm (~10 Hz), what specific biological component would you add first, and why?
> 2. Epileptic seizures are associated with pathologically strong, synchronous population oscillations. Based on what you changed in Exercise 2, which parameter changes in this model pushed the network toward larger, more synchronised oscillations? Do these have a plausible biological analogue (e.g. loss of inhibition)?
> 3. Why is it important, when someone shows you a "beautiful" simulated oscillation, to ask what specific biological mechanism generated it — rather than just accepting that a wiggly, band-limited signal is a validated model of real EEG?

*Your answers here:*

1.
2.
3.

---

### Further reading

- Izhikevich, E.M. (2003). Simple model of spiking neurons. *IEEE Transactions on Neural Networks*, 14(6), 1569-1572.
- Izhikevich, E.M. (2004). Which model to use for cortical spiking neurons? *IEEE Transactions on Neural Networks*, 15(5), 1063-1070.
- Tiesinga, P. & Sejnowski, T.J. (2009). Cortical enlightenment: are attentional gamma oscillations driven by ING or PING? *Neuron*, 63(6), 727-732.
- Buzsáki, G. & Draguhn, A. (2004). Neuronal oscillations in cortical networks. *Science*, 304(5679), 1926-1929.

### Optional extension (advanced)

If you finish early, try one of the following (write your own code cells below):
- Add a brief, strong external stimulus to the excitatory population partway through the simulation (an "evoked" input) and see how the network's oscillatory phase resets — this is the basic idea behind **evoked/event-related** EEG responses.
- Reduce `p_connect` well below 1.0 (sparser, more biologically realistic connectivity) and see how much you need to compensate `exc_strength`/`inh_strength` to recover a similar oscillation, and what that tells you about the trade-off between connection density and synaptic strength in real cortex.
- Split the excitatory population into two groups with a stimulus-selective input to each, and see whether their proxy signals synchronise or compete — a minimal model of binding-by-synchrony.